In [1]:

import pandas as pd
import yaml

    
try:
    with open("../config.yaml", "r") as file:
        cfg = yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

In [2]:
#importing the raw data of energy and co2
raw_df=pd.read_csv(cfg['input_data']['file1'])
df=raw_df.copy()
df.head(1)

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
##to know which data is useful for the project, we read the description of the dataset given per column:
codebook_df=pd.read_csv(cfg['input_data']['file2'])
code_df=codebook_df.copy()
code_df.head(1)

,column,title,description,unit,source
0,country,Country,Geographic location.,NaN,Our World in Data - Regions (2024)


In [4]:
#we take just the columns we definetly will need:
cols=['country','year','iso_code','population','co2','co2_including_luc','co2_growth_prct',
      'co2_including_luc_growth_prct','co2_per_unit_energy','co2_including_luc_per_unit_energy',
     'primary_energy_consumption']

df=df[cols]
df.head(2)

,country,year,iso_code,population,co2,co2_including_luc,co2_growth_prct,co2_including_luc_growth_prct,co2_per_unit_energy,co2_including_luc_per_unit_energy,primary_energy_consumption
0,Afghanistan,1750,AFG,2802560.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
code_df[(code_df['column'].isin(cols))][['column','description','unit']]

,column,description,unit
0,country,Geographic location.,NaN
1,year,Year of observation.,NaN
2,iso_code,ISO 3166-1 alpha-3 three-letter country codes.,NaN
3,population,"Population by country, available from 10,000 B...",people
7,co2,Annual total emissions of carbon dioxide (CO₂)...,million tonnes (Mt)
9,co2_growth_prct,Annual percentage growth in total emissions of...,%
10,co2_including_luc,Annual total emissions of carbon dioxide (CO₂)...,million tonnes (Mt)
12,co2_including_luc_growth_prct,Annual percentage growth in total emissions of...,%
15,co2_including_luc_per_unit_energy,Annual total emissions of carbon dioxide (CO₂)...,kilograms per kilowatt-hour (kg/kWh)
18,co2_per_unit_energy,Annual total emissions of carbon dioxide (CO₂)...,kilograms per kilowatt-hour (kg/kWh)


In [6]:
#Now we only take the data for the timewindow we defined (2014-2024)
df.country.unique()
df=df[(df.year>=2014) & (df.year<=2024)]
df.head(2)

,country,year,iso_code,population,co2,co2_including_luc,co2_growth_prct,co2_including_luc_growth_prct,co2_per_unit_energy,co2_including_luc_per_unit_energy,primary_energy_consumption
264,Afghanistan,2014,AFG,32792528.0,8.697668,8.776041,-2.179384,-1.925987,0.326609,0.329552,26.630217
265,Afghanistan,2015,AFG,33831767.0,9.384400,9.696866,7.895589,10.492492,0.303177,0.313272,30.953520


In [7]:
df = df.rename(columns={
    'population':'pop',
    'co2_including_luc': 'co2_luc',
    'co2_growth_prct':'co2_grow_prct',
    'co2_including_luc_growth_prct':'co2_luc_prct',
    'co2_per_unit_energy':'co2_p_ener',
    'co2_including_luc_per_unit_energy':'co2_luc_p_ener',
    'primary_energy_consumption':'prim_ener_cons'
})
df.info()

<class 'pandas.DataFrame'>
Index: 2794 entries, 264 to 50410
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         2794 non-null   str    
 1   year            2794 non-null   int64  
 2   iso_code        2398 non-null   str    
 3   pop             2519 non-null   float64
 4   co2             2717 non-null   float64
 5   co2_luc         2321 non-null   float64
 6   co2_grow_prct   2563 non-null   float64
 7   co2_luc_prct    2321 non-null   float64
 8   co2_p_ener      2315 non-null   float64
 9   co2_luc_p_ener  2153 non-null   float64
 10  prim_ener_cons  2270 non-null   float64
dtypes: float64(8), int64(1), str(2)
memory usage: 261.9 KB


In [8]:
#where iso_code is null
nulls=df[df.iso_code.isnull()==True]
display(nulls.country.unique())


<StringArray>
[                                  'Africa',
                             'Africa (GCP)',
                                     'Asia',
                               'Asia (GCP)',
             'Asia (excl. China and India)',
                    'Central America (GCP)',
                                   'Europe',
                             'Europe (GCP)',
                     'Europe (excl. EU-27)',
                     'Europe (excl. EU-28)',
                      'European Union (27)',
                      'European Union (28)',
                    'High-income countries',
                   'International aviation',
                   'International shipping',
                                   'Kosovo',
                        'Kuwaiti Oil Fires',
                  'Kuwaiti Oil Fires (GCP)',
 'Least developed countries (Jones et al.)',
                     'Low-income countries',
            'Lower-middle-income countries',
                        'Middle East (GCP

In [9]:
#here we see that nulls are all region groups (continents) and Kosovo is the only country
#then we drop all, except Kosovo as this is not a region but a country.
df = df[(df['iso_code'].notnull()) | (df['country'] == 'Kosovo')]
df.info()

<class 'pandas.DataFrame'>
Index: 2409 entries, 264 to 50410
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         2409 non-null   str    
 1   year            2409 non-null   int64  
 2   iso_code        2398 non-null   str    
 3   pop             2387 non-null   float64
 4   co2             2376 non-null   float64
 5   co2_luc         2134 non-null   float64
 6   co2_grow_prct   2354 non-null   float64
 7   co2_luc_prct    2134 non-null   float64
 8   co2_p_ener      2129 non-null   float64
 9   co2_luc_p_ener  1967 non-null   float64
 10  prim_ener_cons  2139 non-null   float64
dtypes: float64(8), int64(1), str(2)
memory usage: 225.8 KB


In [10]:
#the only country without iso code.
df[df['iso_code'].isnull()].country.unique()

<StringArray>
['Kosovo']
Length: 1, dtype: str

In [11]:
#importing the csv region-mapping from OWD
region_df=pd.read_csv(cfg['input_data']['file3'])
reg_df=region_df.copy()
reg_df.head(1)

,Entity,Code,Year,World region according to OWID
0,Afghanistan,AFG,2023,Asia


In [12]:
reg_df = reg_df.rename(columns={'Entity':'country','World region according to OWID': 'Region'})
reg_df=reg_df.drop(columns=['Code','Year'])
reg_df

,country,Region
0,Afghanistan,Asia
1,Aland Islands,Europe
2,Albania,Europe
3,Algeria,Africa
4,American Samoa,Oceania
...,...,...
267,Yemen Arab Republic,Asia
268,Yemen People's Republic,Asia
269,Yugoslavia,Europe
270,Zambia,Africa


In [13]:
#now we can merge the mapping and our data set.
df = df.merge(reg_df[['country', 'Region']], on='country', how='left')

In [14]:
df=df.rename(columns={'Region':'region'})
df.head(1)

,country,year,iso_code,pop,co2,co2_luc,co2_grow_prct,co2_luc_prct,co2_p_ener,co2_luc_p_ener,prim_ener_cons,region
0,Afghanistan,2014,AFG,32792528.0,8.697668,8.776041,-2.179384,-1.925987,0.326609,0.329552,26.630217,Asia


In [15]:
df[df.region.isnull()==True].country.unique()

<StringArray>
['Antarctica']
Length: 1, dtype: str

In [16]:
df = df[df['country'] != 'Antarctica']
df.info()

<class 'pandas.DataFrame'>
Index: 2398 entries, 0 to 2408
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         2398 non-null   str    
 1   year            2398 non-null   int64  
 2   iso_code        2387 non-null   str    
 3   pop             2387 non-null   float64
 4   co2             2365 non-null   float64
 5   co2_luc         2134 non-null   float64
 6   co2_grow_prct   2354 non-null   float64
 7   co2_luc_prct    2134 non-null   float64
 8   co2_p_ener      2119 non-null   float64
 9   co2_luc_p_ener  1967 non-null   float64
 10  prim_ener_cons  2129 non-null   float64
 11  region          2398 non-null   str    
dtypes: float64(8), int64(1), str(3)
memory usage: 243.5 KB


In [17]:
print(f"Country without iso code: {df[df.iso_code.isnull()==True].country.unique()}")

Country without iso code: <StringArray>
['Kosovo']
Length: 1, dtype: str


In [18]:
df[df.co2.isnull()==True].country.unique()

<StringArray>
['Monaco', 'San Marino', 'Vatican']
Length: 3, dtype: str

In [19]:
df= df[-df['country'].isin(['Monaco', 'San Marino', 'Vatican'])]
df.info()

<class 'pandas.DataFrame'>
Index: 2365 entries, 0 to 2408
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         2365 non-null   str    
 1   year            2365 non-null   int64  
 2   iso_code        2354 non-null   str    
 3   pop             2354 non-null   float64
 4   co2             2365 non-null   float64
 5   co2_luc         2134 non-null   float64
 6   co2_grow_prct   2354 non-null   float64
 7   co2_luc_prct    2134 non-null   float64
 8   co2_p_ener      2119 non-null   float64
 9   co2_luc_p_ener  1967 non-null   float64
 10  prim_ener_cons  2129 non-null   float64
 11  region          2365 non-null   str    
dtypes: float64(8), int64(1), str(3)
memory usage: 240.2 KB


In [20]:
#Now the only missing values are for prim_ener_cons, those are for the countries:
df[df.prim_ener_cons.isnull()==True].country.unique()

<StringArray>
[             'Afghanistan',                  'Albania',
                  'Andorra',                   'Angola',
                 'Anguilla',      'Antigua and Barbuda',
                  'Armenia',                    'Aruba',
                  'Bahamas',                  'Bahrain',
 ...
                  'Tunisia', 'Turks and Caicos Islands',
                   'Tuvalu',                   'Uganda',
                  'Uruguay',                  'Vanuatu',
        'Wallis and Futuna',                    'Yemen',
                   'Zambia',                 'Zimbabwe']
Length: 136, dtype: str

In [21]:
# Save df to the output path defined in config, with reset index
df.to_csv(cfg['output_data']['file1'], index=False)

print(f"File saved to: {cfg['output_data']['file1']}")

File saved to: ../data/clean/energy_co2_data_cleaned.csv
